# Stage 1: Select Position Openers

Select wallets whose opening BUYs are worth copying.
Uses volatility-based wallet metrics + threshold scoring (matching reference notebook).

Grid-search over selection thresholds to maximize **copyable PnL from opening buys** on the validation split.

**Output:** `stage1_result.json` with best selection params.

In [1]:
%load_ext autoreload
%autoreload 2

import time

import numpy as np
import pandas as pd

from lib import (
    load_trades,
    split_data,
    compute_copyable_notional,
    compute_opening_metrics,
    evaluate_wallet_group,
    evaluate_wallet_group_openers,
    select_copyable_group,
    run_grid_search,
    save_stage_result,
    DEFAULT_TAGS,
)
from polymarket_analysis.wallet_selection.volatility import compute_wallet_metrics

pd.options.display.float_format = "{:.4f}".format
pd.options.display.max_rows = 100

## Load data

In [2]:
df_full = load_trades()
df_full = compute_copyable_notional(df_full)
df_train, df_val, df_test = split_data(df_full, method='chronological')

Markets: 1974837
Filtered markets for {'Weather'}: 91123
Loading 16 trade shards...
Total trades loaded: 14,250,603
Unique wallets: 4,082
Date range: 2025-01-09 15:32:39+00:00 -> 2026-07-27 06:12:25+00:00
Chronological split: train <= 2026-05-21T00:00:00Z, val <= 2026-06-23T00:00:00Z, test > 2026-06-23T00:00:00Z
Method: chronological  |  Unique end dates: 112  (train=44, val=33, test=35)

  Train:  4,337,961 trades  (15,923 markets)
  Val:    5,211,194 trades  (20,297 markets)
  Test:   4,701,448 trades  (20,655 markets)
  Total: 14,250,603 trades  (56,875 markets)


In [3]:
df_test['end_date_iso'].min()

'2026-06-24T00:00:00Z'

## Compute wallet metrics on training data

In [4]:
wallet_vol, _ = compute_wallet_metrics(df_train)

wallet_vol["copyable_pnl_factor"] = np.clip(
    wallet_vol["copyable_pnl"] / wallet_vol["total_pnl"].replace(0, np.nan),
    0, 1.0,
).fillna(0.0)
wallet_vol["copyable_roi"] = wallet_vol["average_roi"] * wallet_vol["copyable_pnl_factor"]

opening_metrics = compute_opening_metrics(df_train)
wallet_vol = wallet_vol.merge(opening_metrics, on="wallet", how="left")
for c in ["opening_roi", "opening_pnl", "opening_copyable_roi", "opening_copyable_pnl"]:
    wallet_vol[c] = wallet_vol[c].fillna(0.0)

print(f"Wallets with metrics: {len(wallet_vol)}")
wallet_vol[["wallet", "buy_roi", "opening_roi", "opening_pnl", "opening_copyable_pnl", "copyable_pnl", "num_buckets"]].head(10)

Wallets with metrics: 3220


,wallet,buy_roi,opening_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,num_buckets
0,0x0054ee7dfb882d2d016fa13ef5f5cdb3b0ebcf1f,0.0094,0.0062,9.4227,-0.1185,-0.0009,50
1,0x005ed998fcb786679eb8bfd0d20c15c0903d6d8e,0.0733,0.0681,52.7871,-3.1619,4.8473,210
2,0x00833cc2d777e6f2fc8437679124024ae6468cb1,NaN,-1.0000,-1.0000,0.0000,0.0000,2
3,0x0141be702d272f17666e280303ad44e7bc0cc2da,0.6776,0.6033,59.3074,60.0349,119.3868,50
4,0x015be8bad14c79d2722a0bd8bbe0cd93b905556d,NaN,-0.0877,-17.1649,-20.6348,-20.4954,344
5,0x01a5fb1fa13f378138a31382c8364b5d4e2b0e36,0.0090,0.0194,5.7705,1.4397,6.6597,49
6,0x01a68281185e728ba0fef6245008bf8af68a59b0,0.0139,0.0102,27.6810,-7.3452,-10.3457,1002
7,0x01ced860d8dca5d7987579d2a2635df8520d27a2,0.1658,0.1260,899.2771,95.0644,-88.2798,1478
8,0x01d94480e2a96cdd01fed071878b1adf82e0acd0,NaN,-0.0042,-42.1920,-25.7250,63.4976,1454
9,0x01f2a8baabe17c2541d1e3091220991f257ac3de,0.0099,0.0036,179.5891,-761.1021,-3379.6111,49897


## Baseline selection (reference defaults)

In [5]:
copyable_group = select_copyable_group(
    wallet_vol,
    min_buy_roi=0.05,
    min_num_buckets=20,
    min_num_markets=15,
    max_drawdown_to_pnl=0.20,
    max_top_market_pnl_pct=0.25,
    max_market_pnl_hhi=0.30,
    min_total_notional=5_000,
    min_opening_roi=0.0,
    min_opening_pnl=0,
    min_opening_copyable_roi=0.0,
)
print(f"Copyable group: {len(copyable_group)} wallets")
show_cols = ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
             "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
copyable_group[[c for c in show_cols if c in copyable_group.columns]].head(15)

Copyable group: 29 wallets


,wallet,opening_roi,opening_copyable_roi,opening_pnl,opening_copyable_pnl,copyable_pnl,buy_roi,num_buckets
0,0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3,0.7477,0.7923,581.4098,175.8440,142.6776,0.6066,452
1,0x35aff83368c69c47af04ee2d99330154f22f1ca6,0.6031,0.5008,522.7273,207.6186,260.4761,0.6693,577
2,0x919698b19427cbe6945b0dc823f2d9e126a4d934,0.0948,0.1978,580.1146,203.5358,475.8249,0.1200,961
3,0x8fb431f5057112cfdb0e7f566b4b687cb69cb9ed,0.1696,0.1925,1764.4774,596.3652,3863.1790,0.2845,2762
4,0x07d601375c9bbb9037ad3c7a8f8fa0deff8164fb,0.2075,0.1695,425.9361,61.4549,62.5795,0.1153,478
5,0x4b371de80c78f6484771ff72a961b63566192e48,0.1169,0.1690,347.6881,270.0447,362.5168,0.1536,502
6,0x7231a52f9de4fda5218d0e63f30a3499a4535afe,0.2210,0.1655,526.5890,123.0785,1128.9699,0.3109,1089
7,0x0e09d1f32963451855e429c384be6499dd0e5eef,0.1982,0.1438,404.3683,162.0869,288.0283,0.1949,159
8,0x45e606f7849330adad37875f835fbdd1b7868fca,0.0688,0.1365,543.3619,318.2269,481.4359,0.1437,1096
9,0x46532d38063a22045404aeead6a9f3c49e75fc2b,0.0644,0.1318,264.6895,145.4317,342.5518,0.0636,214


## Baseline evaluation (reference format)

In [6]:
wallet_set = set(copyable_group["wallet"])

for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


*** TRAIN copyable group ***
  Open  : wallet_pnl=  21620.55  roi=0.0992  |  copyable_pnl=   4141.65  roi=0.0856
  Total : wallet_pnl=  59213.89  roi=0.1240  |  copyable_pnl=  16791.98  roi=0.1200

*** VAL copyable group ***
  Open  : wallet_pnl=  16019.56  roi=0.0568  |  copyable_pnl=   1704.47  roi=0.0251
  Total : wallet_pnl=  28838.86  roi=0.0432  |  copyable_pnl=   1256.65  roi=0.0066

*** TEST copyable group ***
  Open  : wallet_pnl=  19389.45  roi=0.0816  |  copyable_pnl=    156.37  roi=0.0030
  Total : wallet_pnl=  27035.25  roi=0.0544  |  copyable_pnl=  -2119.13  roi=-0.0158


## Grid search

Vary selection thresholds to maximize copyable PnL from opening buys on the validation split.

In [7]:
param_grid = dict(
    min_buy_roi=[0.05, 0.07, 0.1],
    min_num_buckets=[15],
    min_num_markets=[10, 20],
    max_drawdown_to_pnl=[0.1, 0.2, 0.3],
    max_top_market_pnl_pct=[1],
    max_market_pnl_hhi=[0.30],
    min_total_notional=[1_000],
    min_opening_roi=[0.05, 0.07, 0.1, 0.2],
    min_opening_pnl=[200],
    min_opening_copyable_roi=[0.05, 0.07, 0.1],
)

print(f"Grid: {np.prod([len(v) for v in param_grid.values()]):.0f} combos")

Grid: 216 combos


In [8]:
res_df = run_grid_search(param_grid, wallet_vol, df_val)
# print(f'open_copyable_pnl: {res_df["open_copyable_pnl"].max():.0f}, open_copyable_roi: {res_df["open_copyable_pnl"].max():.4f}')
res_df.head()

best_row = res_df.iloc[0]
best_params = {k: best_row[k] for k in param_grid.keys()}
best_group = select_copyable_group(wallet_vol, **best_params)

print(f"Best config (val open copyable_pnl={best_row['open_copyable_pnl']:.2f}):")
print(best_params)
print(f"  wallets: {best_row['wallets']:.0f}  open_wallets: {best_row['open_wallets']:.0f}")

Grid: 216 combos, 8 workers
  [100/216] 15.1s elapsed
  [200/216] 29.9s elapsed
  [216/216] 31.6s elapsed
Done: 216 configs in 31.6s
Best config (val open copyable_pnl=3375.31):
{'min_buy_roi': np.float64(0.05), 'min_num_buckets': np.float64(15.0), 'min_num_markets': np.float64(10.0), 'max_drawdown_to_pnl': np.float64(0.3), 'max_top_market_pnl_pct': np.float64(1.0), 'max_market_pnl_hhi': np.float64(0.3), 'min_total_notional': np.float64(1000.0), 'min_opening_roi': np.float64(0.07), 'min_opening_pnl': np.float64(200.0), 'min_opening_copyable_roi': np.float64(0.05)}
  wallets: 53  open_wallets: 41


In [9]:
print('top 10 results')
res_df.head(10)

top 10 results


,min_buy_roi,min_num_buckets,min_num_markets,max_drawdown_to_pnl,max_top_market_pnl_pct,max_market_pnl_hhi,min_total_notional,min_opening_roi,min_opening_pnl,min_opening_copyable_roi,open_copyable_pnl,open_wallet_pnl,open_wallets,total_copyable_pnl,wallets,elapsed
28,0.0500,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,3375.3135,41322.9290,41,-1894.7646,53,1.2008
62,0.0500,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,3342.0950,40771.6728,39,-1959.5323,50,1.2423
101,0.0700,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,3326.3934,41093.2418,40,-1742.1086,52,1.4458
97,0.0700,15,10,0.3000,1,0.3000,1000,0.0500,200,0.0500,3317.0389,41493.9217,42,-1652.1260,54,1.6276
138,0.0700,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,3293.1749,40541.9856,38,-1806.8763,49,1.4494
133,0.0700,15,20,0.3000,1,0.3000,1000,0.0500,200,0.0500,3283.8204,40942.6654,40,-1716.8937,51,1.5309
27,0.0500,15,10,0.3000,1,0.3000,1000,0.0500,200,0.0500,3217.3551,42300.0743,47,-1463.9132,59,1.4287
63,0.0500,15,20,0.3000,1,0.3000,1000,0.0500,200,0.0500,3184.1366,41748.8180,45,-1528.6809,56,1.8004
173,0.1000,15,10,0.3000,1,0.3000,1000,0.0700,200,0.0500,3022.1178,34745.1819,36,-912.0911,48,1.2228
209,0.1000,15,20,0.3000,1,0.3000,1000,0.0700,200,0.0500,3012.2201,34645.5687,35,-925.3220,46,1.3365


## Stage 1 results

In [10]:
if best_group is not None and not best_group.empty:
    print(f"Copyable group: {len(best_group)} wallets")
    cols = [c for c in ["wallet", "opening_roi", "opening_copyable_roi", "opening_pnl",
                         "opening_copyable_pnl", "copyable_pnl", "buy_roi", "num_buckets"]
            if c in best_group.columns]
    print(best_group[cols].head(15).to_string())

Copyable group: 53 wallets
                                        wallet  opening_roi  opening_copyable_roi  opening_pnl  opening_copyable_pnl  copyable_pnl  buy_roi  num_buckets
0   0x74957ea27ac4fbdee46d861fdae357859ff67fcf       0.6199                1.8607    1012.6703              283.2200      327.8016   0.8233         9309
1   0x0a59a7f2a870392a5f555aaa11f49d612e748e5c       0.9370                1.1262    1094.7708              482.7124      477.6809   0.8226         1329
2   0x76305b2a31e7ec35189650c93e8df2a15b92789d       0.6783                0.9857     478.4984              313.4714      504.3710   1.4334          389
3   0xf1e18ec32b2f1e123bc098e3956e6fd00012c152       0.4716                0.8065     744.3432              393.6461     2082.6522   1.3649         1054
4   0xd8798d9aa2c7c05bfefaf6436e8cbd4cd5de7bc3       0.7477                0.7923     581.4098              175.8440      142.6776   0.6066          452
5   0x515ca60f91f49e5e4aa23ab60a6edec3c3917587       0.

In [11]:
wallet_set = set(best_group["wallet"])
print(f"\nSelected {len(wallet_set)} wallets")
for split_name, df_split in [("TRAIN", df_train), ("VAL", df_val), ("TEST", df_test)]:
    evaluate_wallet_group(df_split, wallet_set, label=f"{split_name} copyable group")


Selected 53 wallets

*** TRAIN copyable group ***
  Open  : wallet_pnl=  41181.12  roi=0.2104  |  copyable_pnl=   8783.69  roi=0.1900
  Total : wallet_pnl= 121050.31  roi=0.2111  |  copyable_pnl=  28497.40  roi=0.1795

*** VAL copyable group ***
  Open  : wallet_pnl=  41322.93  roi=0.1152  |  copyable_pnl=   3375.31  roi=0.0488
  Total : wallet_pnl=  92990.20  roi=0.0852  |  copyable_pnl=   8657.33  roi=0.0387

*** TEST copyable group ***
  Open  : wallet_pnl=  38679.46  roi=0.1103  |  copyable_pnl=   4074.93  roi=0.0629
  Total : wallet_pnl=  84481.98  roi=0.0695  |  copyable_pnl=   6786.57  roi=0.0285


## Save stage 1 result

In [12]:
import json
from datetime import datetime, timezone
from pathlib import Path

wallet_cols = [
    "wallet", "buy_roi", "opening_roi", "opening_pnl",
    "opening_copyable_roi", "opening_copyable_pnl",
    "copyable_pnl", "copyable_roi",
    "num_buckets", "num_markets", "total_notional", "total_pnl",
    "max_drawdown_to_pnl", "top_market_pnl_pct", "market_pnl_hhi",
    "wallet_quality",
]
wallet_records = best_group[[c for c in wallet_cols if c in best_group.columns]].to_dict(orient="records")


def _convert(obj):
    if isinstance(obj, (np.integer,)):
        return int(obj)
    if isinstance(obj, (np.floating,)):
        return float(obj)
    if isinstance(obj, np.ndarray):
        return obj.tolist()
    return obj


wallet_records = [{k: _convert(v) for k, v in w.items()} for w in wallet_records]

metadata = {
    "type": "openers",
    "tags": sorted(DEFAULT_TAGS),
    "run_timestamp": datetime.now(timezone.utc).isoformat(),
    "n_wallets_selected": len(best_group),
    "n_wallets_total": len(wallet_vol),
}

payload = {
    "stage": 1,
    "best_params": {k: _convert(v) for k, v in best_params.items()},
    "best_open_copyable_pnl": float(best_row["open_copyable_pnl"]),
    "metadata": metadata,
    "wallets": wallet_records,
}

out_path = Path("stage1_result.json")
with open(out_path, "w") as f:
    json.dump(payload, f, indent=2)
print(f"Saved stage 1 implied result -> {out_path.resolve()}")

Saved stage 1 implied result -> /Users/vobornij/projects/polymarket/notebooks/wallet_selection/stage1_result.json


In [13]:
df = df_test[
    (df_test["wallet"].isin(wallet_set))
    & (df_test["side"] == "BUY") & (df_test["position"] == df_test["quantity"])
    ]
len(df)

30260

In [14]:
df['copyable_pnl'].sum()

np.float64(4074.928692160345)